# Asphere Optimization

In [ ]:
import numpy as np

from optiland import analysis, optic, optimization
from optiland.optimization import minimize

Define a starting lens:

In [ ]:
lens = optic.Optic()

# add surfaces
lens.surfaces.add(index=0, thickness=np.inf)
lens.surfaces.add(
    index=1,
    thickness=7,
    radius=1000,
    material="N-SF11",
    is_stop=True,
    surface_type="even_asphere",
    coefficients=[0, 0, 0],
)
lens.surfaces.add(index=2, thickness=30, radius=-1000)
lens.surfaces.add(index=3)

# set aperture
lens.set_aperture(aperture_type="EPD", value=15)

# add field
lens.fields.set_type(field_type="angle")
lens.fields.add(y=0)

# add wavelength
lens.wavelengths.add(value=0.55, is_primary=True)

# draw lens
lens.draw(num_rays=5)

Define optimization problem:

In [ ]:
problem = optimization.OptimizationProblem()

Add operands (targets for optimization):

In [ ]:
input_data = {
    "optic": lens,
    "surface_number": -1,
    "Hx": 0,
    "Hy": 0,
    "num_rays": 5,
    "wavelength": 0.55,
    "distribution": "hexapolar",
}

# add RMS spot size operand
problem.add_operand(
    operand_type="rms_spot_size",
    target=0,
    weight=1,
    input_data=input_data,
)

Define variables - let radius of curvature vary and the first 3 aspheric coefficients.

In [ ]:
problem.add_variable(lens, "radius", surface_number=1)
problem.add_variable(lens, "radius", surface_number=2)
problem.add_variable(lens, "asphere_coeff", surface_number=1, coeff_number=0)
problem.add_variable(lens, "asphere_coeff", surface_number=1, coeff_number=1)
problem.add_variable(lens, "asphere_coeff", surface_number=1, coeff_number=2)

Check initial merit function value and system properties:

In [ ]:
problem.info()

Run optimization:

In [ ]:
result = minimize(problem, "dls")

Print result summary:

In [ ]:
print(result)

Print merit function value and system properties after optimization:

In [ ]:
problem.info()

Draw final lens:

In [ ]:
lens.draw(num_rays=5)

Viewing the spot diagram indicates that the spot size is indeed minimized and that the lens is diffraction limited.

In [ ]:
spot = analysis.SpotDiagram(lens)
spot.view()